# V02 — Plotly Express Advanced

**Topics:** Facets, animations, bubble charts, sunburst, treemap, choropleth, parallel coordinates, funnel charts.

**Reference:** [Plotly Express docs](https://plotly.com/python/plotly-express/)

**Allowed:** `plotly.express`, `plotly.graph_objects`, `pandas`, `numpy`


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.datasets import fetch_openml, fetch_california_housing

# --- Datasets ---
# Retail
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
retail['Quarter'] = retail['InvoiceDate'].dt.to_period('Q').astype(str)
retail['DayOfWeek'] = retail['InvoiceDate'].dt.day_name()
retail['CustomerID'] = retail['CustomerID'].astype(int)

# Housing
housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]
housing['price_tier'] = pd.qcut(
    housing['medhousval'], q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)
housing['income_tier'] = pd.qcut(
    housing['medinc'], q=3,
    labels=['Low Income', 'Mid Income', 'High Income']
)

# Gapminder-style synthetic dataset for animation
np.random.seed(42)
years = list(range(2015, 2024))
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
regions = ['North America', 'Europe', 'APAC', 'LatAm']
rows = []
base = {'Enterprise': 5000, 'Mid-Market': 2000, 'SMB': 800, 'Consumer': 300}
growth = {'Enterprise': 1.08, 'Mid-Market': 1.12, 'SMB': 1.15, 'Consumer': 1.20}
for yr in years:
    for seg in segments:
        for reg in regions:
            rev = base[seg] * (growth[seg] ** (yr - 2015)) * np.random.uniform(0.85, 1.15)
            cust = int(rev / base[seg] * 100 * np.random.uniform(0.9, 1.1))
            churn = np.random.uniform(0.05, 0.25)
            rows.append({'Year': str(yr), 'Segment': seg, 'Region': reg,
                         'Revenue': round(rev, 0), 'Customers': cust, 'ChurnRate': round(churn, 3)})
saas = pd.DataFrame(rows)

# Funnel data
funnel_data = pd.DataFrame({
    'Stage': ['Website Visits', 'Product Page Views', 'Add to Cart', 'Checkout Started', 'Purchase Complete'],
    'Count': [100000, 45000, 18000, 9500, 4200],
    'Channel': ['Organic'] * 5
})

print(f"Retail: {retail.shape} | SaaS: {saas.shape}")
saas.head()

---
## Exercise 1 — Faceted Scatter: Small Multiples

**Spec:** Compare income vs house value across income tiers using small multiples.
- Use `px.scatter` with `facet_col='income_tier'`
- X: `medinc`, Y: `medhousval`
- Color: `price_tier`, color_discrete_sequence: `px.colors.qualitative.Pastel`
- `facet_col_wrap=3` (all 3 tiers in one row)
- Add trendline: `trendline='ols'`
- Marker size 3, opacity 0.5
- Title: `'Income vs House Value by Income Tier'`
- Remove facet prefix from subplot titles using `fig.for_each_annotation`
- Assign to `fig1`

In [ ]:
# YOUR CODE HERE
fig1 = None

fig1.show()

In [ ]:
# --- ASSERTIONS ---
assert fig1.layout.title.text == 'Income vs House Value by Income Tier'
# Trendline adds extra traces
trace_types = [t.type for t in fig1.data]
assert 'scatter' in trace_types
# Facet annotations should not contain 'income_tier='
annotations = [a.text for a in fig1.layout.annotations if a.text]
assert not any('income_tier=' in a for a in annotations), \
    "Facet prefix must be removed from subplot titles"
# 3 facets
facet_count = sum(1 for a in fig1.layout.annotations
                  if a.text in ['Low Income', 'Mid Income', 'High Income'])
assert facet_count == 3
print("✓ Exercise 1 passed")

**Interpretation:** *(Does the income-value relationship change across tiers? Where is the trendline steepest?)*

---
## Exercise 2 — Faceted Bar: Cross-Dimensional Comparison

**Spec:** SaaS revenue by segment, faceted by region.

Build `seg_region`: group `saas` by `Region` and `Segment`, summing Revenue.
- Use `px.bar`
- X: `Segment`, Y: `Revenue`, Facet: `facet_col='Region'`, `facet_col_wrap=2`
- Color: `Segment`
- `barmode='group'`
- `category_orders={'Segment': ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']}`
- Title: `'Revenue by Segment and Region'`
- Remove facet prefix from titles
- Assign to `fig2`

In [ ]:
seg_region = saas.groupby(['Region', 'Segment'], as_index=False)['Revenue'].sum()

# YOUR CODE HERE
fig2 = None

fig2.show()

In [ ]:
# --- ASSERTIONS ---
assert fig2.layout.title.text == 'Revenue by Segment and Region'
assert fig2.layout.barmode == 'group'
annotations = [a.text for a in fig2.layout.annotations if a.text]
assert not any('Region=' in a for a in annotations), "Facet prefix must be removed"
assert any('North America' in a for a in annotations)
print("✓ Exercise 2 passed")

**Interpretation:** *(Which segment dominates in each region? Where is there the most variation?)*

---
## Exercise 3 — Animated Bubble Chart

**Spec:** Animate SaaS revenue growth over years — Gapminder style.
- Use `px.scatter`
- X: `Customers`, Y: `Revenue`
- Size: `Revenue`, `size_max=60`
- Color: `Segment`
- Animation: `animation_frame='Year'`, `animation_group='Segment'`
- Facet: `facet_col='Region'`, `facet_col_wrap=2`
- `log_x=True` (log scale on x)
- Fix axis ranges so they don't rescale during animation:
  - `range_x=[saas.Customers.min()*0.9, saas.Customers.max()*1.1]`
  - `range_y=[0, saas.Revenue.max()*1.1]`
- Title: `'SaaS Growth by Segment & Region (2015–2023)'`
- Assign to `fig3`

In [ ]:
# YOUR CODE HERE
fig3 = None

fig3.show()

In [ ]:
# --- ASSERTIONS ---
assert fig3.layout.title.text == 'SaaS Growth by Segment & Region (2015–2023)'
# Animation frames present
assert fig3.frames is not None and len(fig3.frames) == len(years)
frame_names = [f.name for f in fig3.frames]
assert '2015' in frame_names and '2023' in frame_names
# Log scale
assert fig3.layout.xaxis.type == 'log'
print("✓ Exercise 3 passed")

**Interpretation:** *(Which segment grows fastest across the animation? Is growth consistent across regions?)*

---
## Exercise 4 — Sunburst Chart: Hierarchical Composition

**Spec:** Show revenue hierarchy: Region → Segment → Year (last year only).

Use `saas[saas['Year'] == '2023']`
- Use `px.sunburst`
- `path=['Region', 'Segment']`
- `values='Revenue'`
- `color='Revenue'`, `color_continuous_scale='RdYlGn'`
- `maxdepth=2`
- Title: `'2023 Revenue Hierarchy: Region → Segment'`
- `branchvalues='total'`
- Assign to `fig4`

In [ ]:
saas_2023 = saas[saas['Year'] == '2023'].copy()

# YOUR CODE HERE
fig4 = None

fig4.show()

In [ ]:
# --- ASSERTIONS ---
assert fig4.layout.title.text == '2023 Revenue Hierarchy: Region → Segment'
assert fig4.data[0].type == 'sunburst'
assert fig4.data[0].branchvalues == 'total'
assert fig4.data[0].maxdepth == 2
# All regions present in labels
labels = list(fig4.data[0].labels)
for reg in regions:
    assert reg in labels, f"Region {reg} missing from sunburst"
print("✓ Exercise 4 passed")

**Interpretation:** *(Which region-segment combination is the largest slice? Which is smallest?)*

---
## Exercise 5 — Treemap: Portfolio View

**Spec:** Treemap of the retail dataset — product hierarchy by revenue.

Build `product_summary`: group retail by `Country` (top 8 by revenue) and `StockCode` (top 5 per country), summing Revenue.
- Use `px.treemap`
- `path=[px.Constant('All Products'), 'Country', 'StockCode']`
- `values='Revenue'`
- `color='Revenue'`, `color_continuous_scale='Blues'`
- Title: `'Product Revenue Treemap: Country → Product'`
- `branchvalues='total'`
- Assign to `fig5`

In [ ]:
top8_countries = retail.groupby('Country')['Revenue'].sum().nlargest(8).index
top5_products = (
    retail[retail['Country'].isin(top8_countries)]
    .groupby(['Country', 'StockCode'])['Revenue'].sum()
    .reset_index()
    .groupby('Country', group_keys=False)
    .apply(lambda x: x.nlargest(5, 'Revenue'))
    .reset_index(drop=True)
)

# YOUR CODE HERE
fig5 = None

fig5.show()

In [ ]:
# --- ASSERTIONS ---
assert fig5.layout.title.text == 'Product Revenue Treemap: Country → Product'
assert fig5.data[0].type == 'treemap'
assert fig5.data[0].branchvalues == 'total'
labels = list(fig5.data[0].labels)
assert 'All Products' in labels
assert 'United Kingdom' in labels
print("✓ Exercise 5 passed")

**Interpretation:** *(Which country-product combinations are the biggest revenue drivers? How concentrated is the top product within its country?)*

---
## Exercise 6 — Choropleth Map

**Spec:** World map of revenue by country.

Build `country_revenue`: group retail by Country, sum Revenue. You'll need to map country names to ISO-3 codes manually for the top countries — build a mapping dict for at least 10 countries.

- Use `px.choropleth`
- `locations`: ISO-3 country codes column
- `color='Revenue'`
- `color_continuous_scale='YlOrRd'`
- `projection='natural earth'`
- Title: `'Global Revenue by Country'`
- `range_color=[0, country_revenue['Revenue'].quantile(0.95)]` (cap color at 95th pct to avoid UK dominating)
- Assign to `fig6`

In [ ]:
country_iso = {
    'United Kingdom': 'GBR', 'Germany': 'DEU', 'France': 'FRA',
    'EIRE': 'IRL', 'Spain': 'ESP', 'Netherlands': 'NLD',
    'Belgium': 'BEL', 'Switzerland': 'CHE', 'Portugal': 'PRT',
    'Australia': 'AUS', 'Norway': 'NOR', 'Sweden': 'SWE',
    'Denmark': 'DNK', 'Japan': 'JPN', 'Finland': 'FIN'
}

country_revenue = (
    retail.groupby('Country')['Revenue'].sum()
    .reset_index()
    .assign(ISO=lambda df: df['Country'].map(country_iso))
    .dropna(subset=['ISO'])
)

# YOUR CODE HERE
fig6 = None

fig6.show()

In [ ]:
# --- ASSERTIONS ---
assert fig6.layout.title.text == 'Global Revenue by Country'
assert fig6.data[0].type == 'choropleth'
assert fig6.layout.geo.projection.type == 'natural earth'
assert fig6.data[0].colorscale is not None
# Range cap applied
cap = country_revenue['Revenue'].quantile(0.95)
assert fig6.data[0].zmax is not None
assert abs(fig6.data[0].zmax - cap) < 1
print("✓ Exercise 6 passed")

**Interpretation:** *(How geographically concentrated is revenue? What risks does this present?)*

---
## Exercise 7 — Parallel Coordinates: Multi-Dimensional Pattern

**Spec:** Explore relationships across all SaaS KPIs simultaneously.

Build `saas_2023_agg`: group 2023 SaaS data by Segment, averaging Revenue, Customers, ChurnRate.
- Use `px.parallel_coordinates`
- Dimensions: `['Revenue', 'Customers', 'ChurnRate']`
- Color: encode Segment as integer (0–3) for color mapping
- `color_continuous_scale=px.colors.diverging.Tealrose`
- Title: `'SaaS KPI Profiles by Segment (2023)'`
- Label axes clearly: `labels={'Revenue': 'Avg Revenue ($)', 'Customers': 'Avg Customers', 'ChurnRate': 'Churn Rate'}`
- Assign to `fig7`

In [ ]:
seg_order = {'Enterprise': 0, 'Mid-Market': 1, 'SMB': 2, 'Consumer': 3}
saas_2023_agg = (
    saas[saas['Year'] == '2023']
    .groupby('Segment', as_index=False)
    .agg({'Revenue': 'mean', 'Customers': 'mean', 'ChurnRate': 'mean'})
)
saas_2023_agg['SegmentCode'] = saas_2023_agg['Segment'].map(seg_order)

# YOUR CODE HERE
fig7 = None

fig7.show()

In [ ]:
# --- ASSERTIONS ---
assert fig7.layout.title.text == 'SaaS KPI Profiles by Segment (2023)'
assert fig7.data[0].type == 'parcoords'
dim_labels = [d.label for d in fig7.data[0].dimensions]
assert 'Avg Revenue ($)' in dim_labels
assert 'Churn Rate' in dim_labels
print("✓ Exercise 7 passed")

**Interpretation:** *(Which segment has the best balance of revenue and churn? What trade-offs are visible in the parallel axes?)*

---
## Exercise 8 — Funnel Chart

**Spec:** E-commerce conversion funnel.
- Use `px.funnel`
- X: `Count`, Y: `Stage`
- Color: `Stage`
- Add a `ConversionRate` column to `funnel_data`: % of first stage (Website Visits)
- Add `StepConversion`: % converted from previous step (first = 100%)
- Use `text=funnel_data['StepConversion'].map(lambda x: f'{x:.1f}%')` on the bars
- Title: `'E-Commerce Conversion Funnel'`
- `color_discrete_sequence=px.colors.sequential.Blues_r`
- Assign to `fig8`

In [ ]:
funnel_data = funnel_data.copy()
funnel_data['ConversionRate'] = (funnel_data['Count'] / funnel_data['Count'].iloc[0] * 100).round(1)
funnel_data['StepConversion'] = (
    funnel_data['Count'] / funnel_data['Count'].shift(1) * 100
).fillna(100).round(1)

# YOUR CODE HERE
fig8 = None

fig8.show()

In [ ]:
# --- ASSERTIONS ---
assert fig8.layout.title.text == 'E-Commerce Conversion Funnel'
assert fig8.data[0].type == 'funnel'
assert len(fig8.data[0].y) == 5
assert fig8.data[0].text is not None
print("✓ Exercise 8 passed")

**Interpretation:** *(Where is the biggest drop-off in the funnel? What would a 5% improvement in that stage mean for final conversions?)*

---
## Exercise 9 — Animated Bar Race

**Spec:** Animate segment revenue ranking over years — a "bar race" chart.

Build `seg_year`: group saas by Year and Segment, summing Revenue. Sort by Year and Revenue.
- Use `px.bar` with `animation_frame='Year'`, `animation_group='Segment'`
- X: `Revenue`, Y: `Segment`
- `orientation='h'`
- Color: `Segment`
- Fix x-axis range to `[0, saas.groupby(['Year','Segment'])['Revenue'].sum().max() * 1.1]`
- Add text labels: `text='Revenue'`, format with `texttemplate='$%{x:,.0f}'`
- Title: `'SaaS Revenue Race by Segment (2015–2023)'`
- Set animation transition duration to 500ms via `fig.layout.updatemenus`
- Assign to `fig9`

In [ ]:
seg_year = (
    saas.groupby(['Year', 'Segment'], as_index=False)['Revenue'].sum()
    .sort_values(['Year', 'Revenue'], ascending=[True, True])
)

# YOUR CODE HERE
fig9 = None

fig9.show()

In [ ]:
# --- ASSERTIONS ---
assert fig9.layout.title.text == 'SaaS Revenue Race by Segment (2015–2023)'
assert fig9.frames is not None and len(fig9.frames) == len(years)
assert fig9.data[0].orientation == 'h'
assert fig9.data[0].texttemplate == '$%{x:,.0f}'
assert fig9.layout.xaxis.range is not None
print("✓ Exercise 9 passed")

**Interpretation:** *(Does segment ranking change over the years? Which segment has the most consistent position?)*

---
## Exercise 10 — Capstone: Multi-View Insight Report

**Spec:** Build a 3-panel insight report for the SaaS dataset using a single function.

Write `saas_insight_report(df)` that produces and returns a dict of 3 figures:
1. `'churn_vs_revenue'`: scatter of avg Revenue vs avg ChurnRate per Segment per Year, colored by Segment, sized by Customers, animated by Year. Title: `'Churn vs Revenue Trade-off Over Time'`.
2. `'regional_share'`: 100% stacked bar of Revenue share by Segment per Region (2023 only). Title: `'2023 Revenue Mix by Region'`.
3. `'growth_index'`: line chart showing revenue indexed to 2015=100 per Segment. Title: `'Revenue Growth Index (2015 = 100)'`.

Assert all 3 are valid Plotly figures with correct titles.

In [ ]:
def saas_insight_report(df: pd.DataFrame) -> dict:
    """
    Returns dict with keys: 'churn_vs_revenue', 'regional_share', 'growth_index'
    """
    # YOUR CODE HERE
    pass

report = saas_insight_report(saas)
report['churn_vs_revenue'].show()
report['regional_share'].show()
report['growth_index'].show()

In [ ]:
# --- ASSERTIONS ---
import plotly.basedatatypes
assert set(report.keys()) == {'churn_vs_revenue', 'regional_share', 'growth_index'}
for key, title in [
    ('churn_vs_revenue', 'Churn vs Revenue Trade-off Over Time'),
    ('regional_share', '2023 Revenue Mix by Region'),
    ('growth_index', 'Revenue Growth Index (2015 = 100)')
]:
    fig = report[key]
    assert isinstance(fig, plotly.basedatatypes.BaseFigure), f"{key} must be a Plotly figure"
    assert fig.layout.title.text == title, f"Wrong title for {key}: {fig.layout.title.text}"

# Growth index: 2015 values must be 100 for all segments
growth_fig = report['growth_index']
for trace in growth_fig.data:
    first_y = trace.y[0] if hasattr(trace.y, '__len__') else None
    if first_y is not None:
        assert abs(first_y - 100) < 1, f"2015 index must be 100 for {trace.name}"

print("✓ Exercise 10 passed")

**Interpretation:** *(Looking at all 3 charts together — which segment would you prioritize for investment? Back your answer with evidence from each chart.)*